# IOAI — 2024 First Stage Pruning (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/base_params.pkl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-pruning/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 가지치기 — 모범답안 (반복 가지치기 + 미세조정)

크기 기반 가지치기를 여러 라운드로 점진 적용하고, 매 라운드 뒤 **0 을 유지한 채** 남은 가중치를 미세조정해 MSE 를 회복한다. 이 데이터는 대부분의 가중치가 중요해 희소도 ~0.5 부근이 최적(그 이상은 MSE 가 1000 초과). score ≈ **0.16**.

## 데이터·베이스 모델 로드

In [ ]:
import pickle, copy, numpy as np, torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)
class MLP(nn.Module):
    def __init__(s):
        super().__init__(); s.flatten = nn.Flatten()
        s.layers = nn.Sequential(nn.Linear(128,1024), nn.Sigmoid(), nn.Linear(1024,10))
    def forward(s, x): return s.layers(s.flatten(x))
# 고정 사전학습 베이스 모델 로드(학습 안 함 — 결정적)
base = MLP().to(device); sd = base.state_dict()
bp = pickle.load(open("data/base_params.pkl","rb"))
for k in sd:
    if k in bp: sd[k] = torch.as_tensor(bp[k])
base.load_state_dict(sd)
Xtr = torch.tensor(np.load("data/X_train.npy"), dtype=torch.float32)
ytr = torch.tensor(np.load("data/y_train.npy"), dtype=torch.float32)
Xva = torch.tensor(np.load("data/X_valid.npy"), dtype=torch.float32).to(device)
yva = torch.tensor(np.load("data/y_valid.npy"), dtype=torch.float32).to(device)
def report(m):
    m.eval()
    with torch.no_grad(): mse = nn.functional.mse_loss(m(Xva), yva).item()
    z=t=0
    for _,p in m.named_parameters(): z+=int((p==0).sum()); t+=p.numel()
    sp=z/t; sc=(1-min(mse,1000)/1000)**1.5*sp**1.5
    print(f"mse {mse:.2f} | sparsity {sp:.3f} | score {sc:.4f}"); return sc
def save_parameters(m):
    pickle.dump({n:p.detach().cpu() for n,p in m.named_parameters()}, open("model_parameters.pkl","wb"))
print("base:", end=" "); report(base)

## 반복 가지치기 + 미세조정(희소도 0.5)

In [ ]:
model = copy.deepcopy(base)
dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
TARGET, ROUNDS, EPOCHS = 0.50, 5, 15
for r in range(ROUNDS):
    frac = TARGET*(r+1)/ROUNDS                        # 점진적으로 희소도↑
    with torch.no_grad():
        for n,p in model.named_parameters():
            if "weight" in n:
                k=int(frac*p.numel())
                if k>0: thr=p.abs().flatten().kthvalue(k).values; p[p.abs()<=thr]=0
    mask={n:(p!=0).float() for n,p in model.named_parameters()}   # 0 유지 마스크
    opt=torch.optim.Adam(model.parameters(),1e-3)
    for e in range(EPOCHS):
        model.train()
        for xb,yb in dl:
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad()
            nn.functional.mse_loss(model(xb),yb).backward(); opt.step()
            with torch.no_grad():
                for n,p in model.named_parameters(): p*=mask[n]     # 잘린 가중치 0 고정
    print(f"round {r} frac {frac:.2f}:", end=" "); report(model)

## 저장 → model_parameters.pkl

In [ ]:
save_parameters(model); print("saved model_parameters.pkl"); report(model)

희소도/미세조정 스케줄·라운드를 조정해 score 를 더 짜낼 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['model_parameters.pkl']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)